# Lab 00 — ตรวจ environment ก่อนเริ่ม
### Setup check

**กลุ่ม:** A พื้นฐาน · **อ่านคู่กับสไลด์หน้า:** – · **เวลาโดยประมาณ:** 15 นาที · **Dataset:** ไม่ใช้

## จุดประสงค์การเรียนรู้
เมื่อทำ lab นี้จบ นิสิตจะสามารถ
1. ยืนยันว่า `uv sync` ติดตั้ง library ครบและ kernel ของ Jupyter ใช้ `.venv` ของ lab
2. รู้ว่า `set_seed` ทำอะไร และทำไมทุก lab ถึงเริ่มด้วยมัน
3. รู้จักโครงสร้างโฟลเดอร์ `lab/` และรู้ว่าจะหา source code ของ `nnlab` ได้ที่ไหน

> **วิธีใช้ notebook นี้:** อ่าน markdown cell ก่อน แล้วรัน code cell ถัดไปด้วย `Shift+Enter` ทีละ cell ตามลำดับ
> ทุก code cell มีคำอธิบายว่าทำอะไรและทำไม ถ้าอยากทดลอง ให้แก้ตัวเลขใน cell แล้วรันซ้ำได้เลย

## ขั้นที่ 1 · library ครบหรือไม่
ถ้า cell ถัดไปพิมพ์ version ของทุก library ได้โดยไม่มี error แสดงว่า environment พร้อมแล้ว
ถ้าขึ้น `ModuleNotFoundError` ให้กลับไปที่ terminal ในโฟลเดอร์ `lab/` แล้วรัน `uv sync` จากนั้นเลือก kernel ใหม่ (Kernel → Change Kernel)

In [1]:
import sys, platform
print("Python", sys.version.split()[0], "บน", platform.system(), platform.machine())
print("interpreter:", sys.executable)          # ควรอยู่ใน lab/.venv/

import numpy, pandas, sklearn, matplotlib, torch, torchvision
for m in (numpy, pandas, sklearn, matplotlib, torch, torchvision):
    print(f"{m.__name__:<12} {m.__version__}")

Python 3.14.7 บน Windows AMD64
interpreter: c:\lab\neural-network-lab\.venv\Scripts\python.exe
numpy        2.5.3
pandas       2.3.3
sklearn      1.9.0
matplotlib   3.11.1
torch        2.14.0+cpu
torchvision  0.29.0+cpu


## ขั้นที่ 2 · package `nnlab`
`nnlab` คือ package ประกอบ lab ที่อยู่ใน `lab/src/nnlab/` — `uv sync` ติดตั้งแบบ *editable* หมายความว่าถ้าเราแก้ไฟล์ใน `src/nnlab/`
แล้ว restart kernel ก็จะเห็นผลทันทีโดยไม่ต้องติดตั้งใหม่
ทุก lab จะเขียนโค้ดเองก่อน แล้วค่อยเทียบกับเวอร์ชันใน `nnlab` ที่จัดระเบียบแล้ว (มี docstring, type hints และ test)

In [2]:
import nnlab, pathlib
print("nnlab", nnlab.__version__, "อยู่ที่", pathlib.Path(nnlab.__file__).parent)
print()
print("โมดูลที่มีให้ใช้:")
for p in sorted(pathlib.Path(nnlab.__file__).parent.glob("*.py")):
    if p.name != "__init__.py":
        first_doc = p.read_text(encoding="utf-8").split('"""')[1].strip().splitlines()[0]
        print(f"  {p.stem:<14} {first_doc}")

nnlab 0.1.0 อยู่ที่ C:\lab\neural-network-lab\src\nnlab

โมดูลที่มีให้ใช้:
  activations    activation function และอนุพันธ์ (สไลด์ p.4-6, p.10-12)
  baselines      โมเดลสำเร็จรูปของ scikit-learn ห่อให้มี interface เดียวกับ nnlab.Perceptron / NeuralNetwork
  config         การตั้งค่าการเทรนแบบ production: dataclass เดียว ใช้ได้ทั้งจาก argparse และไฟล์ JSON
  conv           convolution และ pooling เขียนด้วย loop ธรรมดา เพื่อให้เห็นกลไก (สไลด์ p.164-198)
  conventions    สะพานข้าม convention ของข้อมูลระหว่างสไลด์กับ library
  data           โหลด dataset ทุกตัวที่ใช้ใน lab และแบ่งข้อมูลแบบ stratified (สไลด์ p.122-125)
  exercise       ตัวช่วยตรวจแบบฝึกหัดใน notebook: พิมพ์ PASS / FAIL / ยังไม่ได้ทำ แต่ "ไม่ raise" เพื่อให้ notebook รันต่อได้ทั้งไฟล์
  experiment     รันการทดลองหนึ่งครั้งแบบ production: โหลดข้อมูล → สร้างโมเดล → เทรน → ประเมิน → บันทึกทุกอย่างลง runs/
  losses         loss / cost function (สไลด์ p.17-18) และ regularization term (p.84-86)
  metrics        การประเมินผล classi

## ขั้นที่ 3 · seed คืออะไร ทำไมต้องตั้ง
คอมพิวเตอร์ "สุ่ม" ตัวเลขด้วยสูตรที่เริ่มจากค่าตั้งต้นที่เรียกว่า **seed** — seed เดียวกันให้ลำดับตัวเลขเดียวกันเสมอ
neural network เริ่มจาก weight สุ่ม ถ้าไม่ตั้ง seed ตัวเลขทุกอย่างจะเปลี่ยนทุกครั้งที่รัน ทำให้เทียบกับคำอธิบายใน notebook ไม่ได้
`set_seed(463)` ตั้ง seed ให้ทั้ง `random`, `numpy` และ `torch` แล้วคืน `numpy.random.Generator` ที่เราจะใช้สุ่มตลอด lab

In [3]:
from nnlab.utils import set_seed

rng = set_seed(463)
a = rng.normal(size=3)
rng = set_seed(463)          # ตั้ง seed เดิมอีกครั้ง
b = rng.normal(size=3)
print("รอบแรก :", a)
print("รอบสอง :", b)
print("เหมือนกันทุกตัว:", (a == b).all())

รอบแรก : [-0.50152947 -0.28211293 -0.47814905]
รอบสอง : [-0.50152947 -0.28211293 -0.47814905]
เหมือนกันทุกตัว: True


## ขั้นที่ 4 · ข้อมูลที่ lab ใช้
ข้อมูลเล็กอยู่ใน `lab/data/*.csv` (สร้างด้วย `scripts/make_data.py`) ส่วน dataset ของ scikit-learn อยู่ในตัว library
MNIST สำหรับ lab12 จะดาวน์โหลดอัตโนมัติครั้งแรก (~11 MB) — ถ้าอยู่ในห้องที่อินเทอร์เน็ตช้า รัน cell สุดท้ายล่วงหน้าที่บ้านได้

In [4]:
from nnlab.utils import data_dir
from nnlab.data import load_lung_cancer_toy

print("โฟลเดอร์ข้อมูล:", data_dir())
for p in sorted(data_dir().glob("*.csv")):
    print(f"  {p.name:<24} {p.stat().st_size:>7,} bytes")

X, y = load_lung_cancer_toy()
print("\nตัวอย่าง lung_cancer_toy (สไลด์ p.40): X shape", X.shape, "y", y)

โฟลเดอร์ข้อมูล: C:\lab\neural-network-lab\data
  churn_synthetic.csv       23,921 bytes
  churn_toy.csv                119 bytes
  lung_cancer_toy.csv           62 bytes

ตัวอย่าง lung_cancer_toy (สไลด์ p.40): X shape (3, 2) y [0 1 0]


### (ไม่บังคับ) ดาวน์โหลดล่วงหน้าสำหรับ lab12-13
ลบ `#` หน้าบรรทัดแล้วรัน — MNIST (~11 MB) ไปที่ `lab/data/mnist/` และน้ำหนัก ResNet-18 (~45 MB) ไปที่ cache ของ torch
ทำที่บ้านครั้งเดียว ในห้องเรียนจะได้ไม่ต้องรอดาวน์โหลด

In [5]:
# from nnlab.data import load_mnist
# images, labels = load_mnist(train=True); print(images.shape, labels[:10])
# from nnlab.vision import build_resnet18_transfer
# model = build_resnet18_transfer(weights="DEFAULT"); print(model.weights_meta)

## ★ Key Takeaways
- environment ของ lab ถูก pin version ไว้ใน `uv.lock` — ทุกคนได้ตัวเลขเดียวกัน
- `nnlab` เป็น package editable: แก้ไฟล์ใน `src/nnlab/` แล้ว restart kernel ก็ใช้ได้ทันที
- ทุก lab เริ่มด้วย `set_seed(463)` เพื่อให้ผลทำซ้ำได้
- ลำดับการเรียน: lab01-04 พื้นฐาน → lab05-06 perceptron → lab07-09 neural network → lab10 evaluation → lab11-12 CNN

## ลองทำเอง
เพิ่ม cell ใหม่ด้านล่างแล้วลองทำ (ไม่มีเฉลยใน notebook — ใช้ผลจาก cell ก่อนหน้าตรวจคำตอบตัวเอง)

1. เปลี่ยน seed จาก 463 เป็นเลขอื่นแล้วรัน cell ขั้นที่ 3 ใหม่ — ตัวเลขเปลี่ยนไหม แล้วถ้าใช้ seed เดิมสองครั้งล่ะ
2. เปิดไฟล์ `src/nnlab/utils.py` แล้วอ่านฟังก์ชัน `set_seed` — มัน seed อะไรบ้าง 3 อย่าง

In [5]:
from nnlab.utils import set_seed

first = set_seed(7).normal(size=3)
second = set_seed(7).normal(size=3)
other = set_seed(463).normal(size=3)

print("seed 7 ครั้งที่ 1:", first)
print("seed 7 ครั้งที่ 2:", second)
print("seed 463:", other)
print("ใช้ seed เดิมได้ผลเหมือนกัน:", (first == second).all())
print("เปลี่ยน seed แล้วผลเปลี่ยน:", not (first == other).all())
print("set_seed ตั้งค่าให้: random, numpy, torch")

seed 7 ครั้งที่ 1: [ 0.00123015  0.29874554 -0.27413786]
seed 7 ครั้งที่ 2: [ 0.00123015  0.29874554 -0.27413786]
seed 463: [-0.50152947 -0.28211293 -0.47814905]
ใช้ seed เดิมได้ผลเหมือนกัน: True
เปลี่ยน seed แล้วผลเปลี่ยน: True
set_seed ตั้งค่าให้: random, numpy, torch
